In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../Data/Train.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [3]:
X = df.drop("Reached.on.Time_Y.N", axis=1)
y = df["Reached.on.Time_Y.N"]
X = pd.get_dummies(X, drop_first=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

In [6]:
select_pipeline = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=2))
])

In [7]:
combined_features = FeatureUnion([
    ("pca_features", pca_pipeline),
    ("selected_features", select_pipeline)
])

In [8]:
X_combined = combined_features.fit_transform(X_train, y_train)

print("Combined Feature Shape:", X_combined.shape)

Combined Feature Shape: (8799, 4)


In [9]:
model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

In [11]:
model.fit(X_combined, y_train)

print("Training Score:", model.score(X_combined, y_train))

predictions = model.predict(X_combined)

print("First 5 Predictions:", predictions[:5])

Training Score: 0.18744862129229412
First 5 Predictions: [0.37229353 0.38120985 0.27303941 0.50677083 0.27548609]
